# 4.2 Read-only Debug Agent

这个 notebook 继续不接 LLM，也不新增工具。目标是用当前已有的 file/search tools，做一个更像 SWE agent 的只读调试流程。

## 目标

4.1 证明了最小 agent loop：task -> tool call -> observation -> final answer。

4.2 的目标是把这个 loop 扩展成只读 debugging flow：

1. 接收一个 debugging task。
2. 列出目标 workspace 文件。
3. 读取 failing test。
4. 搜索相关实现。
5. 读取实现文件。
6. 生成只读诊断结论。
7. 记录完整 trajectory。

这里仍然不修改文件，不运行 patch，不接 LLM。

## 导入依赖并设置目标 workspace

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, cast


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "prototype" else Path.cwd()
AGENT_SRC_ROOT = PROJECT_ROOT / "src"
TARGET_WORKSPACE_ROOT = PROJECT_ROOT / "fixtures" / "workspaces" / "calculator_bug"

os.environ["SWE_AGENT_JOM_WORKSPACE_ROOT"] = str(TARGET_WORKSPACE_ROOT)

if str(AGENT_SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_SRC_ROOT))

import swe_agent_jom.config.settings as settings_module
import swe_agent_jom.runtime.workspace as workspace_module

# Notebook kernels keep imported modules alive across cell reruns. Updating the
# module globals keeps this notebook deterministic after repeated execution.
settings_module.WORKSPACE_ROOT = TARGET_WORKSPACE_ROOT.resolve()
workspace_module.WORKSPACE_ROOT = settings_module.WORKSPACE_ROOT

from swe_agent_jom.core.tool_result import ErrorResult, SuccessResult, ToolResult
from swe_agent_jom.memory.trajectory import Trajectory
from swe_agent_jom.tools.file_tool import list_files, read_file
from swe_agent_jom.tools.search_tool import search_code

## 小工具函数

这些 helper 只负责从 `ToolResult` 里安全取数据，让下面的 agent flow 更容易读。

In [2]:
def require_success(result: ToolResult) -> dict[str, Any]:
    """Return ToolResult data or raise a readable error."""
    if not result["ok"]:
        error = cast(ErrorResult, result)
        raise RuntimeError(f"Tool failed: {error['error_type']}: {error['message']}")

    success = cast(SuccessResult, result)
    return success["data"]


def tool_result_as_dict(result: ToolResult) -> dict[str, Any]:
    """Cast ToolResult for Trajectory.record_tool_result."""
    return cast(dict[str, Any], result)

## 实现只读调试 agent

这个 agent 是 rule-based 的：它知道这次任务应该按固定顺序查看文件。真正接 LLM 后，可以把这些固定步骤替换成模型决策。

In [3]:
def build_debug_diagnosis(
    *,
    test_content: str,
    source_content: str,
    implementation_path: str,
) -> str:
    """Create a simple read-only diagnosis from observed test/source content."""
    expected_add_behavior = "add(2, 3), 5" in test_content
    suspicious_add_logic = "return a - b" in source_content

    if expected_add_behavior and suspicious_add_logic:
        return (
            "Diagnosis: the failing test expects add(2, 3) to equal 5, "
            f"but {implementation_path} implements add with subtraction: return a - b. "
            "The likely fix is to change it to: return a + b."
        )

    return (
        "Diagnosis incomplete: I inspected the test and implementation, "
        "but did not find the expected add/subtraction pattern."
    )


def run_read_only_debug_agent(task: str) -> dict[str, Any]:
    """Run a fixed read-only debugging flow over calculator_bug workspace."""
    trajectory = Trajectory()
    trajectory.record_user_message(task)

    trajectory.record_assistant_message("I will inspect the target workspace files.")
    list_args = {"path": ".", "pattern": "*.py", "max_results": 20}
    trajectory.record_tool_call("list_files", list_args)
    list_result = list_files(**list_args)
    trajectory.record_tool_result("list_files", tool_result_as_dict(list_result))
    require_success(list_result)

    trajectory.record_assistant_message("I will read the calculator tests first.")
    test_args = {"path": "tests/test_calculator.py", "max_chars": 2000}
    trajectory.record_tool_call("read_file", test_args)
    test_result = read_file(**test_args)
    trajectory.record_tool_result("read_file", tool_result_as_dict(test_result))
    test_data = require_success(test_result)
    test_content = cast(str, test_data["content"])

    trajectory.record_assistant_message("I will search for the add implementation.")
    search_args = {
        "query": "def add",
        "path": "src",
        "pattern": "*.py",
        "max_results": 5,
    }
    trajectory.record_tool_call("search_code", search_args)
    search_result = search_code(**search_args)
    trajectory.record_tool_result("search_code", tool_result_as_dict(search_result))
    search_data = require_success(search_result)
    matches = cast(list[dict[str, Any]], search_data["matches"])

    if not matches:
        final_answer = "Diagnosis failed: could not find the add implementation."
        trajectory.record_final_answer(final_answer)
        return {
            "task": task,
            "final_answer": final_answer,
            "trajectory": trajectory.to_dicts(),
        }

    implementation_path = cast(str, matches[0]["path"])

    trajectory.record_assistant_message("I will read the implementation file.")
    source_args = {"path": implementation_path, "max_chars": 2000}
    trajectory.record_tool_call("read_file", source_args)
    source_result = read_file(**source_args)
    trajectory.record_tool_result("read_file", tool_result_as_dict(source_result))
    source_data = require_success(source_result)
    source_content = cast(str, source_data["content"])

    final_answer = build_debug_diagnosis(
        test_content=test_content,
        source_content=source_content,
        implementation_path=implementation_path,
    )
    trajectory.record_final_answer(final_answer)

    return {
        "task": task,
        "final_answer": final_answer,
        "trajectory": trajectory.to_dicts(),
    }

## 运行只读调试 agent

In [4]:
debug_result = run_read_only_debug_agent(
    "The calculator tests are failing. Find the bug without modifying files."
)

print(debug_result["final_answer"])

Diagnosis: the failing test expects add(2, 3) to equal 5, but src/demo_math/calculator.py implements add with subtraction: return a - b. The likely fix is to change it to: return a + b.


## 查看完整 trajectory

In [5]:
print(json.dumps(debug_result["trajectory"], ensure_ascii=False, indent=2))

[
  {
    "kind": "user_message",
    "content": {
      "content": "The calculator tests are failing. Find the bug without modifying files."
    },
    "created_at": "2026-07-08T08:11:22.368228+00:00"
  },
  {
    "kind": "assistant_message",
    "content": {
      "content": "I will inspect the target workspace files."
    },
    "created_at": "2026-07-08T08:11:22.368232+00:00"
  },
  {
    "kind": "tool_call",
    "content": {
      "tool_call_id": null,
      "tool_name": "list_files",
      "arguments": {
        "path": ".",
        "pattern": "*.py",
        "max_results": 20
      }
    },
    "created_at": "2026-07-08T08:11:22.368235+00:00"
  },
  {
    "kind": "tool_result",
    "content": {
      "tool_call_id": null,
      "tool_name": "list_files",
      "result": {
        "ok": true,
        "data": {
          "path": ".",
          "pattern": "*.py",
          "count": 3,
          "max_results": 20,
          "truncated": false,
          "entries": [
            {
  

## 这个 notebook 没做什么

它没有：

- 修改文件
- 应用 patch
- 自动运行测试
- 使用 LLM 做决策

它只证明：现有工具已经能支持一个只读 SWE debugging trajectory。